# mortalityAdherence — Python quick start

Tests of adherence of mortality/annuity tables to pension fund data, following de Melo, Graziadei & Targino (2026), *Annuity Table Validation in Pension Funds* (SSRN 6668699).

**How to use:** run each cell in order (Shift+Enter).

## 1. Install the package (only once per session)

In [ ]:
!pip install -q "mortalityAdherence @ https://github.com/eduardoflm/mortalityAdherence/archive/refs/heads/main.zip#subdirectory=python"

In [ ]:
import pandas as pd
import mortalityAdherence as ma
ma.listTables()

## 2. Load data (long format: year, age, exposed, deaths)

Example data shipped with the package (3 years, ages 40–90):

In [ ]:
data = pd.read_csv("https://raw.githubusercontent.com/eduardoflm/mortalityAdherence/main/python/examples/example_fund_data.csv")
data.head()

**To use your own file in Colab**, run the cell below and choose the file (CSV or Excel). Then adjust the column names in `loadFundData`.

In [ ]:
# from google.colab import files
# up = files.upload()
# name = list(up)[0]
# data = ma.loadFundData(name, sep=';', dec=',',
#                        col_year='Ano', col_age='Idade',
#                        col_exposed='Expostos', col_deaths='Obitos')

## 3. Run the 11 tests

In [ ]:
result = ma.testAdherence(data, table="AT-2000m", alpha=0.05)
print(result)

In [ ]:
result  # styled table

In [ ]:
result.summary

## 4. Save the results

In [ ]:
result.to_excel("results.xlsx")
result.to_html("results.html")
# In Colab, download with:
# from google.colab import files; files.download("results.xlsx")

## 5. Test an adjusted (or external) table

Any DataFrame with columns `age` and `qx` works.

In [ ]:
tab = ma.loadTable("AT-2000m")
tab["qx"] = tab["qx"] * 0.86   # 14% deloaded
print(ma.testAdherence(data, table=tab, quiet=True))

## 6. Compare all built-in tables at once

In [ ]:
rows = []
for name in ma.listTables()["name"]:
    r = ma.testAdherence(data, table=name, quiet=True)
    s = r.summary.set_index("test")
    rows.append({"table": name, "A/E": round(r.ae_ratio, 3),
                 "rejections": int((s["reject_h0"] == "Yes").sum()),
                 "p WaldI_P": s.loc["WaldI_P", "p_value"],
                 "p WaldII_P": s.loc["WaldII_P", "p_value"]})
pd.DataFrame(rows)

## 7. Sample without deaths

When D = 0 the Wald tests are not applicable (paper, Sec. 3.5); the LRT tests remain valid.

In [ ]:
zero = pd.DataFrame({"year": 2024, "age": range(40, 56), "exposed": 800, "deaths": 0})
print(ma.testAdherence(zero, table="AT-2000m", quiet=True))